# Chapter 04. Naive Bayes로 도서 카테고리 분류하기

Chapter 03에서는 도서의 **상품명**을 CountVectorizer와 TF-IDF로 숫자 벡터로 바꾸는 방법을 학습했습니다. 이번 Chapter에서는 그 벡터를 머신러닝 모델에 입력해 **도서 제목으로 분야를 예측하는 텍스트 분류의 전체 흐름**을 학습합니다.

전체 흐름은 다음과 같습니다.

**데이터 준비 → X와 y 정의 → train/test 분리 → train에 TF-IDF fit → train/test transform → Multinomial Naive Bayes 학습 → test 예측 → 성능 확인 → 오분류 확인 → 새 제목 예측**

이번 Chapter의 가장 중요한 원칙은 다음 한 문장입니다.

> **원본 데이터를 먼저 train/test로 나누고, TF-IDF는 train 데이터에만 fit합니다.**

이번 Notebook도 앞 Chapter들과 같은 방식으로 작성합니다.

**해야 할 일 이해 → AI에게 질문 → AI 답변 확인 → 주석이 달린 코드 실행 → 출력값 직접 확인 → 결과 검증 → Markdown 정리**

이번 Chapter의 목표는 최고 성능을 만드는 것이 아니라 **텍스트 분류 파이프라인의 순서를 정확하게 이해하는 것**입니다.

## 학습 목표

이번 Chapter가 끝나면 다음 내용을 본인의 말로 설명할 수 있어야 합니다.

- 지도학습과 분류의 의미
- 입력 **X**와 정답 **y**
- train/test를 나누는 이유
- `fit()`과 `transform()`의 차이
- TF-IDF를 train에만 fit해야 하는 이유
- Multinomial Naive Bayes의 역할
- Accuracy와 분야별 평가 지표
- 오분류 사례를 확인하는 이유
- 새로운 제목을 예측하는 방법

특히 이번 Chapter에서는 **데이터 누수(data leakage)**를 피하기 위해 순서를 지키는 것이 중요합니다.

## 실습 1. 라이브러리 준비

### AI에게 질문

> Python과 머신러닝을 처음 배우고 있습니다.  
> pandas와 scikit-learn을 사용해서 도서 제목으로 분야를 분류하려고 합니다.  
> 다음 기능에 필요한 import 코드를 작성해 주세요.
>
> - pandas
> - train_test_split
> - TfidfVectorizer
> - MultinomialNB
> - accuracy_score
> - classification_report
> - confusion_matrix
>
> 초보자가 이해할 수 있도록 각각 무엇에 사용하는지도 설명해 주세요.

### AI 답변

이번 Chapter에서는 pandas로 데이터를 다루고, scikit-learn으로 데이터를 분리하고 TF-IDF를 만들고 Naive Bayes 모델을 학습한 뒤 평가합니다.

또한 현재 Notebook 커널에 scikit-learn이 설치되어 있는지도 확인합니다.

In [ ]:
# 현재 Notebook이 실제로 사용하는 Python 환경을 확인합니다.
import sys

print("Python 실행 파일:")
print(sys.executable)

print("\nPython 버전:")
print(sys.version)

# pandas를 불러옵니다.
import pandas as pd

# train/test 분리를 위한 함수입니다.
from sklearn.model_selection import train_test_split

# 텍스트를 TF-IDF 숫자 벡터로 바꾸는 도구입니다.
from sklearn.feature_extraction.text import TfidfVectorizer

# 이번 Chapter에서 사용할 분류 모델입니다.
from sklearn.naive_bayes import MultinomialNB

# 모델 평가에 사용할 함수들입니다.
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Notebook에서 표와 Markdown을 보기 좋게 출력할 때 사용합니다.
from IPython.display import display, Markdown

# scikit-learn 버전도 확인합니다.
import sklearn
print("\nscikit-learn 버전:", sklearn.__version__)

### 실습 1 결과 확인 및 정리

이 셀이 오류 없이 실행되면 이번 Chapter에 필요한 기본 라이브러리를 사용할 준비가 된 것입니다.

특히 현재 환경에서는 **터미널의 Python과 Notebook 커널의 Python이 서로 다를 수 있으므로** `sys.executable`을 먼저 확인하는 습관이 중요합니다.

만약 `ModuleNotFoundError: No module named 'sklearn'`이 발생하면 현재 Notebook이 사용하는 Python에 scikit-learn을 설치해야 합니다.

Notebook 코드 셀에서 다음처럼 설치할 수 있습니다.

```python
import sys
!{sys.executable} -m pip install scikit-learn
```

설치 후에는 Kernel을 재시작한 뒤 다시 import합니다.

## 실습 2. 데이터 불러오기

### AI에게 질문

> Chapter 01에서 만든 `book_bestseller_clean.csv`를 pandas로 불러오고 싶습니다.  
> 현재 프로젝트 루트는 `C:\dev\llm-data-analysis-course`이고, 파일은 `notebooks/book-text-ml` 폴더 안에 있습니다.
>
> 다음 내용을 확인하는 코드를 작성해 주세요.
>
> 1. 파일 존재 여부  
> 2. 데이터 크기  
> 3. 컬럼 이름  
> 4. 상품명과 분야 앞의 10행  
>
> CSV는 utf-8-sig 인코딩으로 저장했습니다.

### AI 답변

프로젝트 루트를 기준으로 상대경로를 사용하면 됩니다. 파일이 존재하는지 먼저 확인한 뒤 데이터를 읽고, 이번 분류에 필요한 `상품명`과 `분야` 컬럼을 확인합니다.

In [ ]:
# 파일 경로를 다루기 위해 Path를 불러옵니다.
from pathlib import Path

# 현재 프로젝트 루트에서 실행하는 것을 기준으로 한 경로입니다.
DATA_PATH = Path("notebooks/book-text-ml/book_bestseller_clean.csv")

# 파일이 실제로 존재하는지 먼저 확인합니다.
print("CSV 파일 존재:", DATA_PATH.exists())
print("CSV 경로:", DATA_PATH)

# CSV 파일을 DataFrame으로 불러옵니다.
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

# 전체 크기와 컬럼을 확인합니다.
print("\n데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())

# 이번 분류에 필요한 두 컬럼을 확인합니다.
df_books[["상품명", "분야"]].head(10)

### 실습 2 결과 확인 및 정리

이번 분류에서 가장 중요한 컬럼은 두 개입니다.

- **상품명** → 모델이 보고 판단할 입력 데이터
- **분야** → 모델이 맞혀야 할 정답

파일이 열렸다고 바로 다음 단계로 넘어가지 않고 다음을 직접 확인합니다.

- `상품명`과 `분야` 컬럼이 실제로 존재하는가?
- 한글이 정상적으로 보이는가?
- 데이터 크기가 Chapter 01 결과와 크게 다르지 않은가?
- 분야 값이 예상한 도서 분류 이름으로 보이는가?

입력 데이터가 잘못되면 뒤의 모델 결과도 의미가 없어지므로 먼저 데이터부터 확인합니다.

## 실습 3. 모델링 데이터 정리하기

### AI에게 질문

> DataFrame `df_books`에서 상품명과 분야 두 컬럼만 사용해 `df_model`을 만들고 싶습니다.
>
> 다음 조건을 만족하는 초보자용 pandas 코드를 작성해 주세요.
>
> 1. 상품명과 분야만 copy  
> 2. 결측치는 빈 문자열로 처리  
> 3. 문자열로 변환  
> 4. 앞뒤 공백 제거  
> 5. 상품명 또는 분야가 빈 행은 제외  
> 6. 인덱스를 다시 0부터 정리  
> 7. 전처리 전후 행 수 비교

### AI 답변

입력 X와 정답 y에 빈 값이 남아 있으면 모델링 과정에서 문제가 생길 수 있으므로, 먼저 두 컬럼만 따로 복사해 정리합니다.

In [ ]:
# 이번 모델링에서 필요한 두 컬럼만 복사합니다.
df_model = df_books[["상품명", "분야"]].copy()

# 전처리 전 행 수를 저장합니다.
rows_before = len(df_model)

# 상품명과 분야를 같은 방식으로 정리합니다.
for col in ["상품명", "분야"]:
    df_model[col] = (
        df_model[col]
        .fillna("")      # 결측치는 빈 문자열로 바꿉니다.
        .astype(str)     # 문자열로 통일합니다.
        .str.strip()     # 앞뒤 공백을 제거합니다.
    )

# 상품명 또는 분야가 빈 행은 제외합니다.
df_model = df_model[
    (df_model["상품명"] != "") &
    (df_model["분야"] != "")
].reset_index(drop=True)

# 전처리 후 행 수를 확인합니다.
rows_after = len(df_model)

print("전처리 전 행 수:", rows_before)
print("전처리 후 행 수:", rows_after)
print("제외된 행 수:", rows_before - rows_after)

# 최종 모델링 데이터 크기를 확인합니다.
print("모델링 데이터 크기:", df_model.shape)

df_model.head(10)

### 실습 3 추가 검증

전처리가 의도대로 되었는지 결측치와 빈 문자열을 다시 확인합니다.

In [ ]:
# 결측치가 남아 있는지 확인합니다.
print("상품명 결측치:", df_model["상품명"].isna().sum())
print("분야 결측치:", df_model["분야"].isna().sum())

# 빈 문자열이 남아 있는지 확인합니다.
print("빈 상품명 존재:", (df_model["상품명"] == "").any())
print("빈 분야 존재:", (df_model["분야"] == "").any())

# X와 y에 사용할 두 컬럼을 다시 눈으로 확인합니다.
display(df_model[["상품명", "분야"]].head(10))

### 실습 3 결과 확인 및 정리

이 단계에서는 모델이 사용할 입력과 정답을 깨끗하게 정리했습니다.

중요한 점은 `상품명`과 `분야`를 **같은 DataFrame에서 함께 필터링**했다는 것입니다. 이렇게 하면 이후 X와 y를 만들 때 서로 길이가 달라지는 문제를 줄일 수 있습니다.

전처리 후에는 결측치와 빈 문자열이 없는지 다시 확인합니다. 단순히 전처리 코드를 실행하는 것보다 **처리 결과를 검증하는 습관**이 중요합니다.

## 실습 4. 분야 분포 확인하기

### AI에게 질문

> 텍스트 분류를 하기 전에 `df_model["분야"]`의 클래스 분포를 확인하고 싶습니다.
>
> 다음 내용을 확인하는 코드를 작성해 주세요.
>
> 1. 분야 종류 수  
> 2. 분야별 데이터 개수 전체  
> 3. 비율(%)  
> 4. 데이터가 2개 미만인 분야  
>
> stratify=y를 사용할 예정이므로 작은 클래스가 있는지도 확인하고 싶습니다.

### AI 답변

분류 문제에서는 클래스별 데이터 수가 크게 다를 수 있으므로 train/test 분리 전에 분포를 먼저 확인합니다. `stratify=y`를 사용하려면 적어도 각 클래스가 분할 가능한지 확인해야 합니다.

In [ ]:
# 분야 종류 수를 확인합니다.
print("분야 종류 수:", df_model["분야"].nunique())

# 분야별 데이터 개수를 계산합니다.
class_counts = df_model["분야"].value_counts()

# 분야별 비율도 계산합니다.
class_ratio = (
    df_model["분야"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# 개수와 비율을 하나의 표로 만듭니다.
class_distribution = pd.DataFrame({
    "개수": class_counts,
    "비율(%)": class_ratio,
})

display(class_distribution)

In [ ]:
# stratify를 사용하기 전에 데이터가 지나치게 적은 클래스를 확인합니다.
small_classes = class_counts[class_counts < 2]

print("데이터가 2개 미만인 분야 수:", len(small_classes))

if len(small_classes) > 0:
    print("\n확인이 필요한 분야:")
    display(small_classes)
else:
    print("모든 분야에 최소 2개 이상의 데이터가 있습니다.")

### 실습 4 결과 확인 및 정리

분야별 데이터 수가 크게 다르면 **클래스 불균형(class imbalance)**이 있을 수 있습니다.

이때 바로 데이터를 삭제하거나 억지로 같은 수로 맞추지 않습니다. 먼저 분포를 확인하고, 뒤에서 **분야별 precision / recall / F1-score와 오분류 사례**를 함께 봅니다.

또한 `stratify=y`는 train/test에 분야 비율을 가능한 비슷하게 유지하는 데 도움이 되지만, 데이터가 지나치게 적은 클래스가 있으면 분할 과정에서 오류가 날 수 있습니다.

따라서 `small_classes`가 비어 있는지 반드시 확인합니다.

## 실습 5. X와 y 정의하기

### AI에게 질문

> 지도학습에서 X와 y가 무엇인지 처음 배우고 있습니다.  
> 이번 데이터에서는 상품명으로 분야를 예측하려고 합니다.
>
> 1. X와 y를 어떻게 정의하는지  
> 2. 각각 어떤 의미인지  
> 3. 길이가 같은지 확인하는 코드  
> 4. 실제 앞의 5개 값을 나란히 확인하는 코드
>
> 를 작성해 주세요.

### AI 답변

이번 문제에서는 **상품명**이 모델이 보고 판단하는 입력 X이고, **분야**가 모델이 맞혀야 하는 정답 y입니다.

In [ ]:
# 모델 입력 X: 도서 제목
X = df_model["상품명"]

# 정답 y: 도서 분야
y = df_model["분야"]

print("X 길이:", len(X))
print("y 길이:", len(y))
print("X와 y 길이가 같은가?:", len(X) == len(y))

# 실제 입력과 정답을 나란히 확인합니다.
xy_preview = pd.DataFrame({
    "X_상품명": X.head(5),
    "y_분야": y.head(5),
})

xy_preview

### 실습 5 결과 확인 및 정리

지도학습은 **입력과 정답이 함께 있는 데이터**로 관계를 학습합니다.

예를 들어:

```text
상품명: 파이썬 데이터 분석 입문
분야: 컴퓨터/IT
```

이라면 모델링 관점에서는:

```text
X = "파이썬 데이터 분석 입문"
y = "컴퓨터/IT"
```

입니다.

이번 문제는 여러 분야 중 하나를 예측하므로 **분류(classification)** 문제입니다.

## 실습 6. train/test 분리하기

### AI에게 질문

> X는 상품명, y는 분야입니다.  
> train_test_split을 사용해 80%는 학습용, 20%는 평가용으로 나누고 싶습니다.
>
> 조건:
> - test_size=0.2
> - random_state=42
> - stratify=y
>
> 분리 후 train/test 크기와 분야 비율을 비교하는 코드도 작성해 주세요.

### AI 답변

train은 모델이 공부할 데이터이고 test는 학습에 사용하지 않은 평가용 데이터입니다. `stratify=y`를 사용하면 분야 비율을 가능한 비슷하게 유지할 수 있습니다.

In [ ]:
# train/test를 나눌 수 있는지 먼저 확인합니다.
if len(small_classes) > 0:
    raise ValueError(
        "데이터가 2개 미만인 분야가 있습니다. "
        "stratify=y를 사용하기 전에 실습 4에서 작은 클래스를 먼저 확인하세요."
    )

# 원본 텍스트와 정답을 먼저 train/test로 나눕니다.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train X:", X_train.shape)
print("Test X :", X_test.shape)
print("Train y:", y_train.shape)
print("Test y :", y_test.shape)

print("\n전체 행 수:", len(X))
print("Train + Test:", len(X_train) + len(X_test))

In [ ]:
# 전체 / train / test의 분야 비율을 비교합니다.
split_ratio_compare = pd.concat(
    [
        y.value_counts(normalize=True).rename("전체"),
        y_train.value_counts(normalize=True).rename("Train"),
        y_test.value_counts(normalize=True).rename("Test"),
    ],
    axis=1,
).fillna(0).mul(100).round(2)

split_ratio_compare.head(20)

### 실습 6 결과 확인 및 정리

- `test_size=0.2` → 약 20%를 평가용으로 사용
- `random_state=42` → 같은 조건에서 같은 분할을 재현
- `stratify=y` → 전체 분야 비율을 train/test에 가능한 비슷하게 유지

중요한 것은 **TF-IDF를 만들기 전에 원본 텍스트 상태에서 먼저 train/test를 나눴다는 점**입니다.

이 순서가 다음 실습의 데이터 누수 방지와 직접 연결됩니다.

## 실습 7. 가장 중요한 원칙 — split을 먼저 한다

### AI에게 질문

> 왜 전체 데이터에 TF-IDF를 먼저 fit한 뒤 train/test로 나누면 안 되나요?  
> 데이터 누수라는 개념을 초보자가 이해할 수 있게 설명해 주세요.  
> 올바른 순서와 잘못된 순서를 비교해 주세요.

### AI 답변

잘못된 순서:

```text
전체 데이터에 TF-IDF fit
→ train/test 분리
→ 모델 학습과 평가
```

올바른 순서:

```text
원본 텍스트
→ train/test 분리
→ train에 TF-IDF fit
→ test는 transform만 수행
→ 모델 학습과 평가
```

TF-IDF의 `fit()`은 단어 사전과 IDF 규칙을 데이터에서 학습합니다. 전체 데이터에 먼저 fit하면 test에 있는 단어 정보도 기준을 만드는 데 사용됩니다.

그러면 test가 완전히 처음 보는 데이터가 아니게 되어 평가를 신뢰하기 어려워질 수 있습니다.

### 실습 7 결과 확인 및 정리

이번 Chapter에서 가장 중요한 규칙입니다.

**Vectorizer도 학습 과정의 일부**입니다.

따라서 모델만 train 데이터로 학습하면 되는 것이 아니라, TF-IDF의 단어 사전과 IDF도 train 데이터로만 학습해야 합니다.

이 원칙은 뒤의 코드에서:

```python
tfidf.fit_transform(X_train)
tfidf.transform(X_test)
```

형태로 구현됩니다.

## 실습 8. fit과 transform 이해하기

### AI에게 질문

> TfidfVectorizer에서 fit, transform, fit_transform의 차이를 작은 예제로 보여 주세요.  
> 특히 train에는 fit_transform, test에는 transform만 사용한다는 점을 코드로 확인하고 싶습니다.

### AI 답변

- **fit** → train 데이터에서 단어 사전과 IDF 규칙을 학습
- **transform** → 이미 학습한 규칙으로 문장을 숫자로 변환
- **fit_transform** → fit과 transform을 한 번에 수행

작은 예제로 실제 shape와 vocabulary를 확인해 보겠습니다.

In [ ]:
# 원리를 확인하기 위한 작은 예제입니다.
demo_train = [
    "파이썬 데이터 분석",
    "머신러닝 입문",
]

demo_test = [
    "파이썬 머신러닝",
]

demo_tfidf = TfidfVectorizer()

# train에는 fit_transform을 사용합니다.
demo_train_matrix = demo_tfidf.fit_transform(demo_train)

# test에는 이미 학습된 기준으로 transform만 사용합니다.
demo_test_matrix = demo_tfidf.transform(demo_test)

print("학습된 단어:", demo_tfidf.get_feature_names_out())
print("Train shape:", demo_train_matrix.shape)
print("Test shape :", demo_test_matrix.shape)

### 실습 8 직접 확인

Train과 Test의 **행 수는 다를 수 있지만 열 수는 같습니다.**

왜냐하면 Test도 Train에서 학습한 **같은 단어 공간(feature space)**을 사용하기 때문입니다.

Test에 새로운 단어가 있어도 Train vocabulary에 없으면 새로운 열을 만들지 않습니다. 이것이 새 데이터마다 다시 fit하지 않는 이유입니다.

## 실습 9. TF-IDF 변환하기

### AI에게 질문

> train/test 분리가 끝났습니다.  
> TfidfVectorizer를 만들고 X_train에는 fit_transform, X_test에는 transform을 적용해 주세요.
>
> 다음도 확인하고 싶습니다.
>
> 1. Train/Test TF-IDF shape  
> 2. 두 행렬의 열 수가 같은지  
> 3. vocabulary 단어 수  
> 4. 첫 20개 feature

### AI 답변

한 개의 TfidfVectorizer 객체를 만들고 train에서 기준을 학습한 뒤 test에는 같은 기준을 적용합니다.

In [ ]:
# TF-IDF Vectorizer를 만듭니다.
tfidf = TfidfVectorizer()

# Train 데이터에서 단어 사전과 IDF를 학습하면서 벡터로 바꿉니다.
X_train_tfidf = tfidf.fit_transform(X_train)

# Test 데이터에는 Train에서 학습한 기준으로 transform만 적용합니다.
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)

# 두 행렬은 같은 feature 공간을 사용하므로 열 수가 같아야 합니다.
print(
    "Train/Test 열 수가 같은가?:",
    X_train_tfidf.shape[1] == X_test_tfidf.shape[1]
)

# 학습된 feature를 확인합니다.
feature_names = tfidf.get_feature_names_out()

print("학습된 단어 수:", len(feature_names))
print("처음 20개 feature:")
print(feature_names[:20])

In [ ]:
# 행 수가 원래 train/test 텍스트 수와 같은지도 확인합니다.
print(
    "Train 행 수 일치:",
    X_train_tfidf.shape[0] == len(X_train)
)

print(
    "Test 행 수 일치:",
    X_test_tfidf.shape[0] == len(X_test)
)

# 희소 행렬 타입도 확인합니다.
print("Train matrix type:", type(X_train_tfidf))
print("Test matrix type :", type(X_test_tfidf))

### 실습 9 결과 확인 및 정리

Train과 Test의 행 수는 다르지만 **열 수는 반드시 같아야 합니다.**

같은 `tfidf` 객체가 만든 동일한 단어 공간을 사용하기 때문입니다.

Test에만 등장하는 새로운 단어가 vocabulary에 없을 수 있는데 이것은 정상입니다. 새 데이터가 들어올 때마다 Vectorizer를 다시 fit하면 학습 때와 다른 좌표계를 만들게 됩니다.

## 실습 10. Multinomial Naive Bayes 이해하기

### AI에게 질문

> TF-IDF 텍스트 분류에서 Multinomial Naive Bayes를 처음 사용합니다.  
> 수식보다 직관적으로 설명해 주세요.
>
> 특히:
> - 모델이 무엇을 학습하는지
> - 왜 텍스트 분류 baseline으로 사용하기 좋은지
> - 지금 단계에서 꼭 알아야 할 정도만 설명해 주세요.

### AI 답변

초보자 단계에서는 다음 흐름으로 이해하면 충분합니다.

```text
어떤 단어 패턴이
어떤 분야에서 자주 나타나는지 학습
        ↓
새 제목의 단어 패턴을 보고
가능성이 높은 분야를 예측
```

Multinomial Naive Bayes는 구조가 비교적 단순하고 학습이 빠르며 희소한 텍스트 벡터와 함께 사용하기 좋아 **baseline 텍스트 분류 모델**로 사용하기 좋습니다.

In [ ]:
# 아직 학습하지 않은 빈 모델 객체를 만듭니다.
model = MultinomialNB()

print(model)

### 실습 10 결과 확인 및 정리

이 셀에서는 아직 모델이 아무것도 학습하지 않았습니다.

`MultinomialNB()`는 모델의 틀만 만든 상태입니다. 실제 학습은 다음 실습의 `model.fit()`에서 시작합니다.

**Vectorizer의 fit과 모델의 fit은 서로 다른 학습입니다.**

- TF-IDF fit → 단어 사전과 IDF 학습
- Naive Bayes fit → TF-IDF 특징과 분야 라벨의 관계 학습

## 실습 11. 모델 학습과 예측

### AI에게 질문

> TF-IDF 변환이 끝났습니다.  
> MultinomialNB를 train 데이터로 학습하고 test 데이터를 예측한 뒤,
> 상품명 / 실제 분야 / 예측 분야를 하나의 DataFrame으로 만들어 주세요.
>
> 예측 개수가 test 데이터와 같은지도 확인해 주세요.

### AI 답변

`model.fit()`에는 Train TF-IDF와 y_train만 사용하고, `model.predict()`에는 Test TF-IDF를 넣습니다.

In [ ]:
# Train 데이터로 모델을 학습합니다.
model.fit(
    X_train_tfidf,
    y_train,
)

# Test 데이터의 분야를 예측합니다.
y_pred = model.predict(X_test_tfidf)

print("Test 데이터 수:", len(X_test))
print("예측 결과 수:", len(y_pred))
print("개수가 같은가?:", len(X_test) == len(y_pred))

# 정답과 예측을 한 표로 만듭니다.
result = pd.DataFrame({
    "상품명": X_test.reset_index(drop=True),
    "실제_분야": y_test.reset_index(drop=True),
    "예측_분야": y_pred,
})

result.head(20)

### 실습 11 결과 확인 및 정리

`model.fit()`에는 **train 데이터만** 사용했습니다.

`y_pred`는 test 제목 각각에 대해 모델이 예측한 분야입니다.

결과표를 만들면 단순한 점수만 보는 것이 아니라 **어떤 제목을 맞혔고 어떤 제목을 틀렸는지** 직접 읽을 수 있습니다.

## 실습 12. Accuracy 확인하기

### AI에게 질문

> y_test와 y_pred를 이용해 accuracy를 계산하고 싶습니다.  
> 맞은 개수와 전체 test 개수도 같이 확인할 수 있게 코드를 작성해 주세요.
>
> Accuracy 하나만으로 충분하지 않은 이유도 설명해 주세요.

### AI 답변

Accuracy는 전체 test 데이터 중 정답을 맞힌 비율입니다. 직접 맞은 개수를 세어 계산 결과와 비교하면 더 이해하기 쉽습니다.

In [ ]:
# sklearn 함수로 Accuracy를 계산합니다.
accuracy = accuracy_score(
    y_test,
    y_pred,
)

# 실제로 맞힌 개수를 직접 셉니다.
correct_count = (y_test.reset_index(drop=True) == y_pred).sum()
test_count = len(y_test)

print("정답 수:", correct_count)
print("Test 전체 수:", test_count)
print(f"Accuracy: {accuracy:.4f}")

# 직접 계산한 값과 sklearn 결과가 같은지도 확인합니다.
manual_accuracy = correct_count / test_count

print(f"직접 계산한 Accuracy: {manual_accuracy:.4f}")
print("두 값이 같은가?:", abs(accuracy - manual_accuracy) < 1e-12)

### 실습 12 결과 확인 및 정리

Accuracy는 이해하기 쉬운 지표입니다.

예를 들어 Accuracy가 0.80이라면 test 데이터의 약 80%를 맞혔다는 의미입니다.

하지만 분야별 데이터 수가 크게 다르면 Accuracy 하나만으로는 충분하지 않을 수 있습니다. 데이터가 많은 분야를 잘 맞히고 데이터가 적은 분야를 거의 못 맞혀도 전체 Accuracy는 높게 보일 수 있기 때문입니다.

그래서 다음 실습에서 **precision / recall / F1-score**를 분야별로 함께 확인합니다.

## 실습 13. Classification Report 확인하기

### AI에게 질문

> classification_report로 분야별 precision, recall, F1-score를 확인하고 싶습니다.  
> zero_division=0을 사용하고, 문자열 출력뿐 아니라 output_dict=True 결과를 DataFrame으로도 보고 싶습니다.
>
> 각 지표의 의미도 초보자 기준으로 설명해 주세요.

### AI 답변

분야별 성능을 보기 위해 classification report를 출력하고, 표 형태로도 확인합니다.

In [ ]:
# 사람이 읽기 쉬운 텍스트 형태의 report를 만듭니다.
report = classification_report(
    y_test,
    y_pred,
    zero_division=0,
)

print(report)

In [ ]:
# 같은 결과를 dictionary로 받아 DataFrame으로 확인합니다.
report_dict = classification_report(
    y_test,
    y_pred,
    zero_division=0,
    output_dict=True,
)

report_df = pd.DataFrame(report_dict).T

report_df

### 실습 13 결과 확인 및 정리

핵심 지표는 다음처럼 이해합니다.

- **precision** → 모델이 그 분야라고 예측한 것 중 실제로 맞은 비율
- **recall** → 실제 그 분야 데이터 중 모델이 찾아낸 비율
- **F1-score** → precision과 recall을 함께 고려한 지표
- **support** → test 데이터에서 실제 해당 분야 샘플 수

모든 숫자를 외우는 것이 목적이 아닙니다.

**어떤 분야는 잘 맞히고 어떤 분야는 성능이 낮은지** 차이를 확인하는 것이 중요합니다.

## 실습 14. Confusion Matrix 확인하기

### AI에게 질문

> confusion_matrix를 만들고 싶습니다.  
> model.classes_ 순서를 사용해서 실제 분야는 행, 예측 분야는 열인 DataFrame으로 보여 주세요.
>
> 대각선과 대각선 밖 숫자의 의미도 설명해 주세요.

### AI 답변

Confusion Matrix는 어떤 실제 분야가 어떤 분야로 예측되었는지 개수를 보여 줍니다.

In [ ]:
# 모델이 학습한 클래스 순서를 사용합니다.
labels = model.classes_

# Confusion Matrix를 계산합니다.
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
)

# 읽기 쉬운 DataFrame으로 바꿉니다.
df_cm = pd.DataFrame(
    cm,
    index=labels,
    columns=labels,
)

print("Confusion Matrix 크기:", df_cm.shape)
df_cm

In [ ]:
# 대각선 합은 올바르게 예측한 개수입니다.
diagonal_correct = cm.diagonal().sum()

print("대각선 합:", diagonal_correct)
print("Accuracy에서 센 정답 수:", correct_count)
print("두 값이 같은가?:", diagonal_correct == correct_count)

### 실습 14 결과 확인 및 정리

Confusion Matrix에서는:

- **행** → 실제 분야
- **열** → 예측 분야
- **대각선** → 맞게 예측한 데이터
- **대각선 밖** → 다른 분야로 잘못 예측한 데이터

분야가 너무 많으면 전체 행렬을 한눈에 읽기 어렵습니다. 이 경우 다음 실습처럼 **실제 오분류 데이터 자체를 읽는 것**이 더 이해하기 쉽습니다.

## 실습 15. 오분류 사례 확인하기

### AI에게 질문

> result DataFrame에서 실제_분야와 예측_분야가 다른 행만 골라 `misclassified`를 만들고 싶습니다.
>
> 다음도 같이 확인해 주세요.
>
> 1. 오분류 수  
> 2. 오분류 비율  
> 3. 앞의 20개 사례  
> 4. 실제→예측 조합 중 자주 나오는 경우

### AI 답변

틀린 사례를 직접 읽는 것은 모델 개선의 출발점입니다. 제목만으로 구분하기 어려운지, 여러 분야에서 공통으로 쓰는 단어가 있는지 등을 확인할 수 있습니다.

In [ ]:
# 실제 분야와 예측 분야가 다른 행만 선택합니다.
misclassified = result[
    result["실제_분야"] != result["예측_분야"]
].copy()

print("오분류 수:", len(misclassified))
print("Test 전체 수:", len(result))
print(
    "오분류 비율:",
    round(len(misclassified) / len(result), 4)
)

misclassified.head(20)

In [ ]:
# 어떤 실제 분야가 어떤 분야로 자주 잘못 예측되었는지 확인합니다.
misclassified_pairs = (
    misclassified
    .groupby(["실제_분야", "예측_분야"])
    .size()
    .reset_index(name="오분류_개수")
    .sort_values("오분류_개수", ascending=False)
)

misclassified_pairs.head(20)

### 실습 15 결과 확인 및 정리

오분류 제목을 직접 읽으면서 다음 질문을 합니다.

- 제목만으로 분야를 판단하기 어려운가?
- 여러 분야에서 사용할 수 있는 단어인가?
- 학습 데이터가 부족한 분야인가?
- 분야 라벨이 지나치게 세분화되어 있는가?

모델 점수만 보고 끝내지 않고 **틀린 사례를 실제 데이터로 확인하는 과정**이 중요합니다.

## 실습 16. 예측 결과 저장하기

### AI에게 질문

> result와 misclassified DataFrame을 각각 CSV로 저장하고 싶습니다.
>
> 파일명:
> - chapter04_predictions.csv
> - chapter04_misclassified.csv
>
> utf-8-sig로 저장한 뒤 파일 존재 여부와 다시 읽은 앞부분까지 확인하게 작성해 주세요.

### AI 답변

Notebook 폴더 안에 결과를 저장하고 실제로 다시 읽어서 검증합니다.

In [ ]:
# 결과 파일 경로를 지정합니다.
PREDICTIONS_PATH = Path(
    "notebooks/book-text-ml/chapter04_predictions.csv"
)

MISCLASSIFIED_PATH = Path(
    "notebooks/book-text-ml/chapter04_misclassified.csv"
)

# 전체 test 예측 결과를 저장합니다.
result.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

# 오분류 결과를 저장합니다.
misclassified.to_csv(
    MISCLASSIFIED_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("예측 결과 파일 존재:", PREDICTIONS_PATH.exists())
print("오분류 파일 존재:", MISCLASSIFIED_PATH.exists())

In [ ]:
# 저장한 파일을 다시 읽어 한글과 컬럼이 정상인지 확인합니다.
predictions_check = pd.read_csv(
    PREDICTIONS_PATH,
    encoding="utf-8-sig",
)

misclassified_check = pd.read_csv(
    MISCLASSIFIED_PATH,
    encoding="utf-8-sig",
)

print("예측 결과 행 수:", len(predictions_check))
print("오분류 결과 행 수:", len(misclassified_check))

display(predictions_check.head())
display(misclassified_check.head())

### 실습 16 결과 확인 및 정리

파일 저장은 `to_csv()` 실행으로 끝내지 않습니다.

- 파일이 실제로 생성되었는가?
- 다시 읽을 수 있는가?
- 한글이 깨지지 않는가?
- 컬럼 이름이 정상인가?
- 원래 DataFrame과 행 수가 같은가?

를 확인합니다.

## 실습 17. 새로운 도서 제목 예측하기

### AI에게 질문

> 학습에 사용하지 않은 새로운 제목 3개를 기존 TF-IDF와 Naive Bayes 모델로 예측하고 싶습니다.
>
> 반드시 새 제목에는 `fit_transform()`이 아니라 기존 `tfidf.transform()`을 사용하고, 상품명과 예상 분야를 DataFrame으로 보여 주세요.
>
> 가능하면 predict_proba로 모델이 가장 높게 본 확률도 같이 확인해 주세요.

### AI 답변

새 데이터에서도 **학습 때 만든 같은 TF-IDF 좌표계**를 유지해야 하므로 기존 `tfidf.transform()`을 사용합니다.

In [ ]:
# 학습에 없던 새 제목 예시입니다.
new_titles = [
    "파이썬으로 시작하는 데이터 분석",
    "처음 배우는 주식 투자",
    "마음을 이해하는 심리학",
]

# 중요: 새 제목에서는 fit_transform()을 사용하지 않습니다.
new_vectors = tfidf.transform(new_titles)

# 기존에 학습한 모델로 분야를 예측합니다.
new_predictions = model.predict(new_vectors)

# 예측 결과를 표로 만듭니다.
new_result = pd.DataFrame({
    "상품명": new_titles,
    "예상_분야": new_predictions,
})

new_result

In [ ]:
# predict_proba를 이용해 각 제목에서 가장 높은 예측 확률도 확인합니다.
new_probabilities = model.predict_proba(new_vectors)

# 각 행에서 가장 큰 확률을 가져옵니다.
max_probabilities = new_probabilities.max(axis=1)

new_result_with_prob = new_result.copy()
new_result_with_prob["가장_높은_예측확률"] = max_probabilities

new_result_with_prob

### 실습 17 결과 확인 및 정리

새 제목 예측의 순서는 반드시 다음과 같습니다.

```text
새 제목
→ 학습된 tfidf.transform()
→ 학습된 model.predict()
```

새 제목에서 다시 `fit_transform()`하면 학습 때와 다른 단어 공간을 만들게 되므로 사용하지 않습니다.

또한 모델이 특정 분야로 예측했다고 해서 그 책의 **공식 분야가 확정되었다는 뜻은 아닙니다.**

현재 학습 데이터에서 배운 패턴을 기준으로 모델이 해당 분야를 예측했다는 의미입니다.

## 실습 18. 모델의 한계 이해하기

현재 모델은 주로 **도서 제목**만 사용합니다.

이번 모델에 들어가지 않은 정보는 다음과 같습니다.

- 책 소개
- 목차
- 저자
- 출판사
- 키워드
- 본문
- 독자 리뷰

따라서 제목에 분야 정보가 충분하지 않으면 모델이 구분하기 어렵습니다.

또한 베스트셀러 데이터는 전체 출판 도서를 대표하지 않을 수 있습니다.

### 표현의 차이

좋은 표현:

> 현재 학습 데이터의 패턴을 바탕으로 모델이 컴퓨터/IT 분야로 예측했다.

피해야 할 표현:

> 이 책은 컴퓨터/IT 분야의 책이다.

모델의 예측과 실제 공식 분류는 구분해야 합니다.

## 실습 19. 데이터 누수 최종 점검

이번 Chapter의 핵심 원칙을 코드 상태와 함께 다시 점검합니다.

### 체크 질문

- train/test를 TF-IDF fit보다 먼저 나눴는가?
- `fit_transform()`은 `X_train`에만 적용했는가?
- `X_test`에는 `transform()`만 적용했는가?
- 모델 학습에는 train 데이터만 사용했는가?
- 새 제목에도 기존 Vectorizer의 `transform()`을 사용했는가?

In [ ]:
# 이번 Notebook에서 사용한 객체와 shape를 다시 확인합니다.
print("원본 X:", X.shape)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)

print("\n학습된 TF-IDF feature 수:", len(tfidf.get_feature_names_out()))
print("모델 클래스 수:", len(model.classes_))

print("\n새 제목 벡터 shape:", new_vectors.shape)
print(
    "새 제목도 같은 feature 수 사용:",
    new_vectors.shape[1] == X_train_tfidf.shape[1]
)

### 실습 19 결과 확인 및 정리

이 다섯 가지를 지키는 것이 이번 Chapter의 핵심입니다.

특히 **test나 새 데이터에서 다시 fit하지 않는다**는 원칙을 기억합니다.

평가용 데이터는 학습 과정에서 미리 보지 않은 상태로 남아 있어야 모델 성능을 더 신뢰할 수 있습니다.

## 실습 20. 자주 만나는 오류

### AI에게 질문

> Naive Bayes 텍스트 분류 Notebook에서 자주 발생하는 오류를 초보자가 점검할 수 있게 정리해 주세요.
>
> 특히:
> - stratify 오류
> - empty vocabulary
> - X와 y 길이 불일치
> - 새 제목에서 다시 fit하는 실수
>
> 를 어떻게 확인해야 하는지 코드와 함께 설명해 주세요.

### AI 답변

오류가 나면 전체 코드를 무작정 다시 작성하지 않고 원인을 좁혀 확인합니다.

In [ ]:
# 1. stratify 관련 점검
print("가장 작은 클래스 개수:")
print(y.value_counts().tail(20))

# 2. empty vocabulary 관련 점검
print("\nX_train 길이:", len(X_train))
print("X_train 앞의 5개:")
print(X_train.head(5).tolist())

# 3. X와 y 길이 확인
print("\nX 길이:", len(X))
print("y 길이:", len(y))
print("같은가?:", len(X) == len(y))

### 실습 20 오류별 확인 방법

**1. stratify 오류**

클래스에 데이터가 너무 적으면 `stratify=y`에서 오류가 발생할 수 있습니다.

```python
y.value_counts().tail(20)
```

으로 작은 클래스를 먼저 확인합니다.

무조건 `stratify=None`으로 바꾸기보다 **왜 작은 클래스가 생겼는지와 데이터 수를 먼저 확인**합니다.

**2. empty vocabulary**

입력 텍스트가 비어 있거나 전처리/토큰화 조건 때문에 사용할 단어가 없을 수 있습니다.

```python
print(len(X_train))
print(X_train.head(20).tolist())
```

으로 실제 텍스트를 확인합니다.

**3. X와 y 길이 불일치**

X와 y는 같은 `df_model`에서 만드는 것이 안전합니다.

**4. 새 제목에서 다시 fit하는 실수**

잘못된 코드:

```python
tfidf.fit_transform(["새로운 도서 제목"])
```

올바른 코드:

```python
tfidf.transform(["새로운 도서 제목"])
```

## 실습 21. 전체 코드 흐름 다시 확인하기

앞에서는 각 단계를 하나씩 이해했습니다. 이번 셀은 **새로운 내용을 추가하는 것이 아니라 전체 흐름을 한 번에 복습**하기 위한 코드입니다.

코드를 외우기보다 다음 단계 이름으로 읽습니다.

**데이터 → X/y → split → TF-IDF → fit → predict → evaluate → new prediction**

In [ ]:
# 1. 데이터 불러오기
df_check = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

# 2. 상품명과 분야 준비
df_model_check = df_check[["상품명", "분야"]].copy()

for col in ["상품명", "분야"]:
    df_model_check[col] = (
        df_model_check[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

df_model_check = df_model_check[
    (df_model_check["상품명"] != "") &
    (df_model_check["분야"] != "")
].reset_index(drop=True)

# 3. X, y
X_check = df_model_check["상품명"]
y_check = df_model_check["분야"]

# 4. train/test 분리
X_train_check, X_test_check, y_train_check, y_test_check = train_test_split(
    X_check,
    y_check,
    test_size=0.2,
    random_state=42,
    stratify=y_check,
)

# 5. TF-IDF
tfidf_check = TfidfVectorizer()

X_train_tfidf_check = tfidf_check.fit_transform(X_train_check)
X_test_tfidf_check = tfidf_check.transform(X_test_check)

# 6. 모델 학습
model_check = MultinomialNB()
model_check.fit(
    X_train_tfidf_check,
    y_train_check,
)

# 7. 평가
y_pred_check = model_check.predict(X_test_tfidf_check)

accuracy_check = accuracy_score(
    y_test_check,
    y_pred_check,
)

print(f"Accuracy: {accuracy_check:.4f}")

print(
    classification_report(
        y_test_check,
        y_pred_check,
        zero_division=0,
    )
)

# 8. 새 제목 예측
new_title_check = [
    "파이썬으로 시작하는 데이터 분석"
]

new_vector_check = tfidf_check.transform(
    new_title_check
)

print(
    "예상 분야:",
    model_check.predict(new_vector_check)[0]
)

In [ ]:
# 앞에서 단계별로 만든 결과와 요약 코드 결과가 같은지 확인합니다.
print(
    "Accuracy 동일:",
    abs(accuracy - accuracy_check) < 1e-12
)

print(
    "TF-IDF feature 수 동일:",
    X_train_tfidf.shape[1] == X_train_tfidf_check.shape[1]
)

print(
    "예측 개수 동일:",
    len(y_pred) == len(y_pred_check)
)

### 실습 21 결과 확인 및 정리

같은 데이터, 같은 `random_state`, 같은 Vectorizer와 모델 설정을 사용했기 때문에 앞에서 단계별로 만든 결과와 요약 코드의 결과가 같아야 합니다.

이 셀의 목적은 **전체 파이프라인을 한 번에 다시 읽는 것**입니다.

각 줄을 보면서 다음 질문에 답할 수 있어야 합니다.

- 어디에서 데이터가 train/test로 나뉘는가?
- 어디에서 TF-IDF가 fit되는가?
- test에서는 왜 transform만 하는가?
- 어디에서 모델이 학습되는가?
- 어디에서 test 예측을 하는가?
- 새 데이터는 어떤 순서로 예측되는가?

## 실습 22. 결과 정리

수업자료의 결과 정리 형식에 맞춰 실제 실행값을 사용한 Markdown을 만듭니다.

임의의 Accuracy나 오분류 수를 적지 않고 **현재 Notebook에서 계산한 변수 값을 그대로 사용**합니다.

In [ ]:
# 대표 오분류 사례를 최대 3개 가져옵니다.
misclassified_examples = misclassified.head(3)

# Markdown용 오분류 문자열을 만듭니다.
if len(misclassified_examples) == 0:
    misclassified_md = "- 이번 test 분할에서는 오분류 사례가 없습니다."
else:
    lines = []

    for _, row in misclassified_examples.iterrows():
        lines.append(
            f"- **{row['상품명']}**  "
            f"(실제: {row['실제_분야']} / 예측: {row['예측_분야']})"
        )

    misclassified_md = "\n".join(lines)

# 실제 실행 결과를 사용해 Chapter 결과 Markdown을 만듭니다.
chapter04_md = f"""
## Chapter 04 결과

### 데이터

- 입력 X: **상품명**
- 정답 y: **분야**
- 전체 모델링 데이터: **{len(df_model)}개**
- Train: **{len(X_train)}개**
- Test: **{len(X_test)}개**
- Train/Test 비율: 약 **80/20**

### 모델

- 텍스트 벡터화: **TfidfVectorizer**
- 분류 모델: **MultinomialNB**
- TF-IDF feature 수: **{X_train_tfidf.shape[1]}개**

### 평가

- Accuracy: **{accuracy:.4f}**
- 분야별 precision / recall / F1-score는 Classification Report에서 확인했습니다.

### 오분류

- 오분류 수: **{len(misclassified)}개**

{misclassified_md}

오분류 사례는 제목만으로 분야를 판단하기 어렵거나,
여러 분야에서 공통으로 사용할 수 있는 단어가 포함되어 있을 수 있으므로
실제 제목을 직접 읽어보며 원인을 확인해야 합니다.

### 한계

- 현재 모델은 주로 **도서 제목만 사용**합니다.
- 책 소개, 목차, 저자, 출판사, 본문, 독자 리뷰 등은 사용하지 않았습니다.
- 현재 베스트셀러 데이터와 현재 분야 라벨 범위 안에서 평가한 결과입니다.
- 모델의 예측은 공식 도서 분류와 동일한 의미가 아닙니다.
"""

display(Markdown(chapter04_md))

### 실습 22 결과 확인 및 정리

최종 Markdown에서 가장 중요한 것은 **실제로 실행하지 않은 수치를 만들어 쓰지 않는 것**입니다.

이번 셀은 현재 Notebook의 실제 변수에서:

- 모델링 데이터 수
- Train/Test 수
- TF-IDF feature 수
- Accuracy
- 오분류 수
- 대표 오분류 사례

를 직접 가져옵니다.

따라서 Notebook을 처음부터 다시 실행하면 현재 데이터와 현재 분할 결과가 자동으로 반영됩니다.

## 최종 체크리스트

Notebook을 제출하거나 다음 Chapter로 넘어가기 전에 **Kernel Restart → Run All**로 처음부터 끝까지 다시 실행합니다.

- [ ] 상품명과 분야 데이터를 정리했다.
- [ ] X와 y의 역할을 설명할 수 있다.
- [ ] 분야 분포와 작은 클래스를 확인했다.
- [ ] train/test를 먼저 분리했다.
- [ ] TF-IDF는 train에만 fit했다.
- [ ] test에는 transform만 적용했다.
- [ ] MultinomialNB를 train 데이터로 학습했다.
- [ ] test 데이터로 성능을 확인했다.
- [ ] Accuracy뿐 아니라 분야별 지표를 확인했다.
- [ ] Confusion Matrix를 확인했다.
- [ ] 오분류 제목을 직접 확인했다.
- [ ] 예측 결과 CSV를 저장하고 다시 읽어 확인했다.
- [ ] 새 제목에는 기존 Vectorizer의 transform만 사용했다.
- [ ] 모델의 한계를 설명할 수 있다.
- [ ] 실제 실행 결과를 이용한 Markdown을 작성했다.

## 이번 Chapter에서 꼭 기억할 5가지

1. **X는 도서 제목, y는 분야입니다.**
2. **train은 학습용, test는 평가용입니다.**
3. **TF-IDF도 학습 과정이므로 train에만 fit합니다.**
4. **Accuracy 하나보다 분야별 지표와 오분류를 함께 봅니다.**
5. **새 데이터에서는 다시 fit하지 않고 transform 후 predict합니다.**

### 다음 Chapter 연결

Chapter 03에서는:

**도서 제목 → TF-IDF 벡터**

이번 Chapter에서는:

**TF-IDF 벡터 → Naive Bayes → 도서 분야 예측**

다음 Chapter에서는 같은 TF-IDF 벡터를 코사인 유사도에 사용해 **비슷한 도서를 찾는 추천 문제**로 연결합니다.

### Chapter 04 한 문장 정리

**원본 데이터를 먼저 train/test로 나누고, TF-IDF를 train에만 fit한 뒤 Multinomial Naive Bayes로 학습하고, test 성능과 오분류를 함께 검증합니다.**